# LightGBM  Optimizacion Bayesiana

Tarea para el Hogar 04,  versión Gustavo Pasquini: 2026

#### 2.1  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

## Generacion de la clase_ternaria

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Tipe -> Runtime type -> R

In [1]:
require("data.table")

root <- getwd()
if (basename(root) == "ensembles") {
    root <- dirname(root)
}

dataset_file = "RANKS_LAGS_O2.csv"

dataset <- fread(
    file.path(root, "entregas", "dataset", dataset_file)
)

setnames(dataset, "ternaria", "clase_ternaria")

Loading required package: data.table



In [2]:
list.files(file.path(root, "entregas", "dataset"))

[1] "c_01_tern_2_lags_O2.csv"                   
 [2] "c_01_tern_2_lags.csv"                      
 [3] "c_01_tern_2_lf_k6_p4.csv"                  
 [4] "c_01_tern_2_lf_O2_k3_p6.csv"               
 [5] "c_01_tern_2.csv"                           
 [6] "competencia_01_crudo.csv"                  
 [7] "competencia_01_ternaria_lags.csv"          
 [8] "competencia_01_ternaria_lf_k6_perdida4.csv"
 [9] "competencia_01_ternaria.csv"               
[10] "RANKS_LAGS_O2.csv"                         
[11] "RANKS_LF_O2_K6_P4.csv"                     
[12] "versiones_dataset_1.ipynb"                 
[13] "versiones_dataset_2.ipynb"

In [3]:
setorder( dataset, foto_mes, clase_ternaria, numero_de_cliente)
dataset[, .N, list(foto_mes, clase_ternaria)]

foto_mes,clase_ternaria,N
<int>,<chr>,<int>
202103,BAJA+1,1019
202103,BAJA+2,960
202103,continua,160921
202104,BAJA+1,964
202104,BAJA+2,1139
202104,continua,161181
202105,BAJA+1,1143
202105,BAJA+2,870
202105,continua,161755


## 2.2 Optimizacion Hiperparámetros

Esta parte se debe correr con el runtime en lenguaje R Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

### 2.2.1 Inicio

limpio el ambiente de R

In [4]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Fri Oct 09 23:19:02 2026"

In [5]:
# limpio la memoria, conservando las rutas y el nombre del dataset
rm(list = setdiff(ls(all.names = TRUE), c("root", "dataset_file")))
gc(full = TRUE, verbose = FALSE)

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,703811,37.6,1411985,75.5,1411985,75.5
Vcells,1320715,10.1,1454074303,11093.8,1518903387,11588.4


### 2.2.2 Carga de Librerias

Esta parte lleva  demasiados   minutos la primera vez en Google Colab
<br> por suerte a partir del 15-septiembre empezaremos a trabajar con Google Cloud, en donde todas las librerias, de Python R y Julia ya estarán instaladas

In [6]:
# instalo paquetes
install.packages( c("R.utils", "rlist" ) )

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



In [7]:
# Librerias  modernas  para la Bayesian Optimization
install.packages( c("bbotk", "mlr3", "mlr3mbo", "mlr3learners", "DiceKriging") )

Warning message in install.packages(c("bbotk", "mlr3", "mlr3mbo", "mlr3learners", :
“installation of package ‘bbotk’ had non-zero exit status”
Warning message in install.packages(c("bbotk", "mlr3", "mlr3mbo", "mlr3learners", :
“installation of package ‘DiceKriging’ had non-zero exit status”
Warning message in install.packages(c("bbotk", "mlr3", "mlr3mbo", "mlr3learners", :
“installation of package ‘mlr3mbo’ had non-zero exit status”
Warning message in install.packages(c("bbotk", "mlr3", "mlr3mbo", "mlr3learners", :
“installation of package ‘mlr3learners’ had non-zero exit status”
Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



In [8]:
# el LightGBM
install.packages(c("lightgbm"))

Warning message in install.packages(c("lightgbm")):
“installation of package ‘lightgbm’ had non-zero exit status”
Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



In [9]:
file.path(R.home("bin"), "R")

[1] "/home/marco/anaconda3/envs/r-env/lib/R/bin/R"

In [10]:
library(data.table)
library(R.utils)
library(rlist)
library(bbotk)
library(mlr3)
library(mlr3mbo)
library(mlr3learners)
# library(paradox)
library(DiceKriging)
library(lightgbm)

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, warnings


Loading required package: paradox


Attaching package: ‘mlr3’


The following object is masked from ‘package:R.utils’:

    resample


Loading required package: mlr3

### 2.2.3 Definicion de Parametros

aqui debe cargar SU semilla primigenia
<br>recuerde cambiar el numero de experimento en cada corrida nueva

In [11]:
PARAM <- list()
PARAM$experimento <- "PAS-001"
PARAM$semilla_primigenia <-  230047   # La semilla de Laura Pasquini


In [12]:
# training y future
PARAM$training_pct <- 70
PARAM$train <- c(202103, 202104)
PARAM$train_final <- c(202103, 202104)
PARAM$future <- c(202106)
PARAM$cortes <- seq(9000, 15000, by= 500)

In [13]:
# un undersampling de 0.1  toma solo el 10% de los CONTINUA
# undersampling de 1.0  implica tomar TODOS los datos

PARAM$trainingstrategy$undersampling <- 1

In [14]:
PARAM$hyperparametertuning$iteraciones <- 30 # iteraciones bayesianas

In [15]:
# Parametros LightGBM

# parametros fijos del LightGBM que se pisaran con la parte variable de la BO
PARAM$lgbm$param_fijos <-  list(
  boosting= "gbdt", # puede ir  dart  , ni pruebe random_forest
  objective= "binary",
  # metric= "average_precision",
  metric= "auc",
  feature_pre_filter= FALSE,
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  force_row_wise= TRUE,
  deterministic= TRUE,
  verbosity= -100,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L,
  min_gain_to_split= 0,
  lambda_l1= 0.0,
  lambda_l2= 0.0,
  max_bin= 31L,

  bagging_fraction= 1.0,
  pos_bagging_fraction= 1.0,
  neg_bagging_fraction= 1.0,
  is_unbalance= FALSE,
  scale_pos_weight= 1.0,

  drop_rate= 0.1,
  max_drop= 50,
  skip_drop= 0.5,

  extra_trees= FALSE,

  early_stopping= 0,
  min_data_in_leaf= 0,     # fijo, esto es FUNDAMENTAL
  feature_fraction= 0.50,  # fijo
  feature_fraction_bynode= 0.20,
  learning_rate= 0.005,    # fijo

  num_iterations= 20000,     # se modificara
  num_leaves= 1024,          # se modificara
  min_sum_hessian_in_leaf= 1e-06  # se modificara
)


Aqui se definen los hiperparámetros de LightGBM que participan de la Bayesian Optimization

In [16]:
# Aqui se cargan los bordes de los hiperparametros de la BO
#  es vital entender que se optimiza min_sum_hessian_in_leaf y que  min_data_in_leaf se deja fijo en CERO

PARAM$hypeparametertuning$hs <- ps(
  num_iterations= p_int(lower= 64L,  upper= 4096L),
  num_leaves= p_int(lower= 16L,  upper= 2048L),
  min_sum_hessian_in_leaf= p_dbl( lower=1e-06, upper=0.1)
)

In [17]:
# particionar agrega una columna llamada fold a un dataset
#   que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(data, division, agrupa= "", campo= "fold", start= 1, seed= NA) {
  if (!is.na(seed)) set.seed(seed, "L'Ecuyer-CMRG")

  bloque <- unlist(mapply(
    function(x, y) {rep(y, x)},division, seq(from= start, length.out= length(division))))

  data[, (campo) := sample(rep(bloque,ceiling(.N / length(bloque))))[1:.N],by= agrupa]
}

In [18]:
# logueo al archivo BO_log.txt
loguear  <- function( preg, arch=NA, verbose=TRUE )
{
  t0 <- Sys.time()
  reg <- copy( preg )
  if( "feature_contri" %in%  names(reg) ) {
    reg$feature_contri <- reg$feature_contri[1]
  }
  archivo <- arch
  if( is.na(arch) ) archivo <- paste0( folder, substitute( reg), ext )


  if( !file.exists( archivo ) )
  {
    # Escribo los titulos
    linea  <- paste0( "fecha\t",
                      paste( list.names(reg), collapse="\t" ), "\n" )

    cat( linea, file=archivo )
  }

  # escribo el registro
  linea  <- paste0( format(t0, "%Y%m%d.%H%M%S"),  "\t",     # la fecha y hora
                    gsub( ", ", "\t", toString( reg ) ),  "\n" )

  cat( linea, file=archivo, append=TRUE )  # grabo al archivo

  if( verbose )  cat( linea )   # imprimo por pantalla
}

### 2.2.4  Preprocesamiento

In [19]:
# carpeta de trabajo
carpeta_experimento <- file.path(root, "DATA", "EXP", PARAM$experimento)

dir.create(carpeta_experimento, recursive = TRUE, showWarnings = FALSE)
setwd(carpeta_experimento)

In [20]:
# lectura del dataset
dataset <- fread(
    file.path(root, "entregas", "dataset", dataset_file),
    stringsAsFactors = TRUE
)

setnames(dataset, "ternaria", "clase_ternaria")

In [21]:
dataset_train <- dataset[foto_mes %in% PARAM$train]

In [22]:
  particionar(dataset_train,
    division= c(PARAM$training_pct, 100L -PARAM$training_pct),
    agrupa= "clase_ternaria",
    seed= PARAM$semilla_primigenia # aqui se usa SU semilla
  )

In [23]:
# paso la clase a binaria que tome valores {0,1}  enteros
#  BAJA+1 y BAJA+2  son  1,   CONTINUA es 0
#  a partir de ahora ya NO puedo cortar  por prob(BAJA+2) > 1/40

dataset_train[,
  clase01 := ifelse(clase_ternaria %in% c("BAJA+2","BAJA+1"), 1L, 0L)
]

In [24]:
# defino los datos que forma parte del training
# aqui se hace el undersampling de los CONTINUA
# notar que para esto utilizo la SEGUNDA semilla

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset_train[, azar := runif(nrow(dataset_train))]
dataset_train[, training := 0L]

dataset_train[
  foto_mes %in%  PARAM$train &  fold==1 &
    (azar <= PARAM$trainingstrategy$undersampling | clase_ternaria %in% c("BAJA+1", "BAJA+2")),
  training := 1L
]

In [25]:
# los campos que se van a utilizar

campos_buenos <- setdiff(
  colnames(dataset_train),
  c("clase_ternaria", "clase01", "fold", "azar", "training")
)

#### 2.2.4.1  Salsa Magica (dedicada a Laura Pasquini)

Los EDAs que estan haciendo los alumnos de los lunes son pura falopa, y de la mala. El uso de Agentes de la IA solo les empera el daño cerebral, generan pedf larguisimos, que no te dan ganas de leerlos, carentes de insights y alma.
<br>El unico EDA que tiene realmente sentido, en este caso, es comparar la distribución de cada variable del mes donde se entrena 202104 y el mes donde se va a aplicar el modelo 202106, buscando Data Drifting.
<br>Aun con los reales datos del futuro, donde no se tiene la clase, SIEMPRE se puede analizar el Data Drifting.
<br>Para las variables con distribuciones distintas se puede hacer:
* Transformar la variable para hacerla homogenea entre los dos periodos
* Eliminarla (es lo simple, que se eligió hacer en este caso)

En esa exploracion gráfica del Data Drifting, se vio que estas variables cambian de distribucion entre 202104 y 202106, y se decidió brutalmente eliminarlas
* cprestamos_personales
* mprestamos_personales


más adelantes, se utilizarán técnicas más inclusivas, y se preservaran dichas variables, transformadas.


In [26]:
campos_buenos <- setdiff( campos_buenos, c("cprestamos_personales", "mprestamos_personales") )

2.2.5 Configuracion Bayesian Optimization

In [27]:
giter <- 0

if( file.exists("BO_log.txt") ){
  tb_BO <- fread("BO_log.txt")
  giter <- tb_BO[, max(iter)]
}


In [28]:
# En el argumento x llegan los parametros de la bayesiana

Estimar_lightgbm <- function(xs) {

  flush.console()
  giter <<- giter + 1

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, xs)
  xx <- copy(param_completo)

  # entreno LightGBM
  modelo <- lgb.train(
    data= dtrain,
    valids= list( "val"=dvalidate),
    param= param_completo
  )


  # obtengo la ganancia
  xx$iter <- giter
  xx$metrica <- modelo$best_score

  loguear( xx, "BO_log.txt")
  set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")  # le reordeno a mlr3MBO

  return( list( metrica = xx$metrica) )
}

In [29]:
# dejo los datos de training en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[training == 1L, campos_buenos, with= FALSE]),
  label= dataset_train[training == 1L, clase01],
  free_raw_data= FALSE
)

nrow(dtrain)
ncol(dtrain)

[1] 228325

[1] 760

In [30]:
# datos de validatin en el formato que necesita LightGBM
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset_train[training == 0L, campos_buenos, with= FALSE]),
  label= dataset_train[training == 0L, clase01],
  free_raw_data= FALSE
)

nrow(dvalidate)
ncol(dvalidate)

[1] 97859

[1] 760

In [31]:
# Aqui comienza la configuracion de la Bayesian Optimization

# en este archivo quedan la evolucion binaria de la BO
kbayesiana <- "bayesiana.RDATA"

funcion_optimizar <- Estimar_lightgbm # la funcion que voy a maximizar

objective <- ObjectiveRFun$new(
  fun= funcion_optimizar,
  domain= PARAM$hypeparametertuning$hs,
  codomain= ps(metrica = p_dbl(tags="maximize")),
  id= "BO_lightgbm"
)


instance <- OptimInstanceBatchSingleCrit$new(
  objective= objective,
  terminator=  trm("evals", n_evals = PARAM$hyperparametertuning$iteraciones)
)


surrogate <- srlrn(lrn("regr.km",
  optim.method = "BFGS",
  nugget.estim = TRUE,
  control= list(trace = FALSE)))

acq_function  <- acqf("ei")                 # expected improvement

acq_optimizer  <- acqo(
  optimizer  = opt("random_search", batch_size= 1000),
  terminator = trm("evals", n_evals= 1000)
)

optimizer <- opt("mbo",
  loop_function = bayesopt_ego,
  surrogate= surrogate,
  acq_function= acq_function,
  acq_optimizer= acq_optimizer
)



2.2.6 Corrida Bayesian Optimization

In [32]:
bayesiana_salida <- optimizer$optimize(instance)

INFO  [23:19:42.693] [bbotk] Starting to optimize 3 parameter(s) with '<OptimizerMbo>' and '<TerminatorEvals> [n_evals=30, k=0]'
INFO  [23:19:42.749] [bbotk] Evaluating 12 configuration(s)
20261009.232045	gbdt	binary	auc	FALSE	FALSE	TRUE	TRUE	TRUE	-100	230047	-1	0	0	0	31	1	1	1	FALSE	1	0.1	50	0.5	FALSE	0	0	0.5	0.2	0.005	640	676	0.0464152212548051	1	0.924814927606078
20261009.232846	gbdt	binary	auc	FALSE	FALSE	TRUE	TRUE	TRUE	-100	230047	-1	0	0	0	31	1	1	1	FALSE	1	0.1	50	0.5	FALSE	0	0	0.5	0.2	0.005	3286	1366	0.00444592912255071	2	0.932477215734691
20261009.233227	gbdt	binary	auc	FALSE	FALSE	TRUE	TRUE	TRUE	-100	230047	-1	0	0	0	31	1	1	1	FALSE	1	0.1	50	0.5	FALSE	0	0	0.5	0.2	0.005	3836	316	0.000194058640758553	3	0.929498080913761
20261009.233257	gbdt	binary	auc	FALSE	FALSE	TRUE	TRUE	TRUE	-100	230047	-1	0	0	0	31	1	1	1	FALSE	1	0.1	50	0.5	FALSE	0	0	0.5	0.2	0.005	880	74	0.00259274381462525	4	0.924035795910764
20261009.233341	gbdt	binary	auc	FALSE	FALSE	TRUE	TRUE	TRUE	-100	230047	-1	0	0	0	31	1	1	1	

Luego de este punto, en la carpeta del experimento quedó el archivo BO_log.txt que posee todas las iteraciones de la Bayesian Optimization

## 2.3  Produccion

### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

In [33]:
setwd(carpeta_experimento)

In [34]:
 # cergo los mejores hiperparametros, que son los de mayor metrica
 tb_bayesiana <- fread("BO_log.txt")
 setorder( tb_bayesiana, -metrica )
 PARAM$out$lgbm$mejores_hiperparametros <- as.list(tb_bayesiana[1])

#### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la  optimización bayesiana

In [35]:
# clase01
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

In [36]:
dataset_train <- dataset[foto_mes %in% PARAM$train_final]
dataset_train[,.N, list(foto_mes, clase_ternaria)]

foto_mes,clase_ternaria,N
<int>,<fct>,<int>
202103,continua,160921
202104,continua,161181
202104,BAJA+2,1139
202103,BAJA+2,960
202104,BAJA+1,964
202103,BAJA+1,1019


#### Final Training Hyperparameters

In [37]:
param_final <- modifyList(PARAM$lgbm$param_fijos,
  PARAM$out$lgbm$mejores_hiperparametros)

param_final

$boosting
[1] "gbdt"

$objective
[1] "binary"

$metric
[1] "auc"

$feature_pre_filter
[1] FALSE

$first_metric_only
[1] FALSE

$boost_from_average
[1] TRUE

$force_row_wise
[1] TRUE

$deterministic
[1] TRUE

$verbosity
[1] -100

$seed
[1] 230047

$max_depth
[1] -1

$min_gain_to_split
[1] 0

$lambda_l1
[1] 0

$lambda_l2
[1] 0

$max_bin
[1] 31

$bagging_fraction
[1] 1

$pos_bagging_fraction
[1] 1

$neg_bagging_fraction
[1] 1

$is_unbalance
[1] FALSE

$scale_pos_weight
[1] 1

$drop_rate
[1] 0.1

$max_drop
[1] 50

$skip_drop
[1] 0.5

$extra_trees
[1] FALSE

$early_stopping
[1] 0

$min_data_in_leaf
[1] 0

$feature_fraction
[1] 0.5

$feature_fraction_bynode
[1] 0.2

$learning_rate
[1] 0.005

$num_iterations
[1] 3465

$num_leaves
[1] 921

$min_sum_hessian_in_leaf
[1] 0.002888681

$fecha
[1] 20261010

$iter
[1] 16

$metrica
[1] 0.9335139

In [38]:
# aplico el modelo a los datos sin clase
dfuture <- dataset[foto_mes %in% PARAM$future]


In [39]:
campos_buenos <- union(
    campos_buenos,
    c("cprestamos_personales", "mprestamos_personales")
)

In [40]:
# Entreno el modelo sobre todo el periodo

dtrain_final <- lgb.Dataset(
  data= data.matrix(dataset_train[, campos_buenos, with= FALSE]),
  label= dataset_train[, clase01]
)

modelo_final <- lgb.train(
  data= dtrain_final,
  param= param_final
)

In [41]:
# hago el predict en los datos del futuro, y calculo la ganancia

# aplico el modelo a los datos nuevos
prediccion <- predict(
  modelo_final,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)



In [42]:
# calculo la ganancia en los datos ddel futuro para todos los cortes
tb_prediccion <- dfuture[, list(numero_de_cliente, foto_mes,clase_ternaria)]
tb_prediccion[, prob := prediccion ]
tb_prediccion[, gan := ifelse(clase_ternaria == "BAJA+2", 1072500, -27500)]

# lgb.save(modelo_final, "modelo.txt" )
# Dibujo la curva de ganancia acumulada
setorder(tb_prediccion, -prob)
tb_prediccion[, ganancia_acumulada := cumsum(gan)]
tb_prediccion[, pos := sequence(.N)]

for (envios in PARAM$cortes) {
  ganancia <- tb_prediccion[1:envios, sum(gan)]
  cat( envios, "\t", ganancia, "\n")
}

9000 	 380600000 
9500 	 387750000 
10000 	 388300000 
10500 	 384450000 
11000 	 388300000 
11500 	 392150000 
12000 	 397100000 
12500 	 399850000 
13000 	 397100000 
13500 	 394350000 
14000 	 392700000 
14500 	 391050000 
15000 	 391600000 


In [43]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sat Oct 10 01:40:15 2026"